# Assignment 1 — Weather Agent Debugging
## Springer Capital — LLM Final Assessment
### 1. Project Overview
This notebook documents the systematic debugging of a LangGraph Weather Agent that retrieves IP-based location data, fetches current weather from Open-Meteo, and formats a personalized report.

Objectives: correct the original defects, verify graph connectivity and state management, reproduce functional checks, test successful and unsuccessful workflows, and document limitations.

Previously observed results: eight bugs corrected and verified, nine technical improvements, 40 manual functional checks and three integration tests containing 25 verification checks. These are historical results, not outputs of this notebook. Run the cells below to reproduce the validation.

Live validation was blocked by IPAPI HTTP 429. All demonstrations below use synthetic responses, not live weather.

### 2. Environment Setup
Use the project virtual environment (Python 3.12.9 in the original development environment). Install the agent requirements and `ipykernel` before running. The following cell locates the assignment directory whether the kernel starts in the repository root or the assignment folder. No packages are installed automatically.

In [13]:
from pathlib import Path
import sys
from importlib.metadata import version
import unittest
from unittest.mock import Mock, patch
import requests
from datetime import timedelta

cwd = Path.cwd().resolve()
assignment_dir = next((p for p in (cwd, *cwd.parents, cwd / "assignment_1")
                       if (p / "weather_agent" / "graph.py").is_file()
                       and (p / "tests" / "test_weather_agent.py").is_file()), None)
if assignment_dir is None:
    raise FileNotFoundError("Open the notebook from the repository or assignment_1 directory.")
agent_dir = assignment_dir / "weather_agent"
sys.path.insert(0, str(agent_dir))
from graph import weather_agent
from components.config import config
from components.state import WeatherAgentState
from components.helper_functions import (
    classify_temperature, get_weather_description, get_greeting,
    parse_utc_offset, format_local_time
)
print("Python:", sys.version.split()[0])
for package in ("langgraph", "langchain", "pydantic", "requests", "ipykernel"):
    print(f"{package}: {version(package)}")

Python: 3.12.9
langgraph: 0.6.4
langchain: 0.3.26
pydantic: 2.11.5
requests: 2.34.2
ipykernel: 7.4.0


### 3. Agent Architecture
### Workflow Overview

The Weather Agent implements a sequential LangGraph workflow consisting of three functional nodes.

Each node performs a specific responsibility and updates the shared agent state.

```mermaid
flowchart TD
    START([START]) --> LOCATION[fetch_location_data]
    LOCATION --> WEATHER[fetch_weather_data]
    WEATHER --> GENERATE[generate_weather_info]
    GENERATE --> END([END])
```

### Node Responsibilities

#### Node 1 — fetch_location_data

Responsibilities:

- Send an HTTP request to the IPAPI service.

- Retrieve approximate geographical information based on the outgoing IP address.

- Validate the presence of required location fields.

- Store the retrieved information in the shared agent state.

Output:

`location_data`

#### Node 2 — fetch_weather_data

Responsibilities:

- Retrieve geographical coordinates from the shared state.

- Construct the Open-Meteo API request.

- Request current weather information using GMT as the reference timezone.

- Validate the presence of required meteorological fields.

- Store the weather information in the shared agent state.

Output:

`weather_data`

#### Node 3 — generate_weather_info

Responsibilities:

- Retrieve location and meteorological information.

- Generate a personalized greeting.

- Classify the current temperature.

- Translate WMO weather codes into readable descriptions.

- Convert UTC time into local time.

- Construct the final formatted weather report.

Output:

`weather_info`

### Shared Agent State

The workflow uses a TypedDict-based state schema named `WeatherAgentState`.

| State Field | Data Type | Responsibility |
|---|---|---|
| name | str | User's name |
| location_data | Optional dictionary | Geographical information |
| weather_data | Optional dictionary | Meteorological information |
| weather_info | Optional string | Final formatted weather report |

### State Evolution

```mermaid
flowchart TD
    A["Initial State: name"]
    B["Location Data Added"]
    C["Weather Data Added"]
    D["Weather Report Generated"]
    A --> B
    B --> C
    C --> D
```

The graph was explicitly compiled using:

```python
weather_agent = builder.compile()
```

Graph compilation and connectivity were successfully verified during the validation phase.

---

In [14]:
print("Graph type:", type(weather_agent).__name__)
assert callable(weather_agent.invoke)
print(weather_agent.get_graph().draw_mermaid())

Graph type: CompiledStateGraph
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	fetch_location_data(fetch_location_data)
	fetch_weather_data(fetch_weather_data)
	generate_weather_info(generate_weather_info)
	__end__([<p>__end__</p>]):::last
	__start__ --> fetch_location_data;
	fetch_location_data --> fetch_weather_data;
	fetch_weather_data --> generate_weather_info;
	generate_weather_info --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### 4. Bug Analysis and Fixes
Eight primary bugs were identified and addressed during the debugging process.

### BUG-001 — Incorrect Application Entry Point

**File:** main.py

**Category:** Application Execution

**Original Code:**

```python
if __name__ == "_main_":
    main()
```

**Root Cause:**

The Python execution guard contained an incorrect string literal. Python assigns `__main__` to the `__name__` variable when a module is executed directly.

Consequently, the application failed to invoke its main function after successful module initialization.

**Correction:**

```python
if __name__ == "__main__":
    main()
```

**Impact:**

The application now executes its main function when launched directly.

**Validation Status:** Verified through application execution.

---

### BUG-002 — Disconnected Weather Retrieval Node

**File:** graph.py

**Category:** Workflow Architecture

**Original Code:**

```python
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)
```

**Root Cause:**

The `fetch_weather_data` node was registered but was not connected to the execution path.

The report generation node could therefore be reached without first retrieving meteorological information.

**Correction:**

```python
builder.add_edge(START, "fetch_location_data")
builder.add_edge("fetch_location_data", "fetch_weather_data")
builder.add_edge("fetch_weather_data", "generate_weather_info")
builder.add_edge("generate_weather_info", END)
```

**Impact:**

The workflow now follows the required sequential data dependencies.

**Validation Status:** Graph connectivity verified through LangGraph introspection.

---

### BUG-003 — Missing Graph Compilation

**File:** graph.py

**Category:** LangGraph Execution

**Original Code:**

```python
weather_agent = builder
```

**Root Cause:**

The StateGraph builder was assigned directly to the application variable without compiling the graph.

The uncompiled builder did not provide the required `invoke()` method.

**Observed Error:**

```text
'StateGraph' object has no attribute 'invoke'
```

**Correction:**

```python
weather_agent = builder.compile()
```

**Impact:**

The application now uses a compiled executable LangGraph workflow.

**Validation Status:** Verified through application execution and graph inspection.

---

### BUG-004 — Incorrect Weather API Endpoint

**File:** components/config.py

**Category:** Configuration Management

**Original Code:**

```python
WEATHER_API_BASE_URL: str = ""
```

**Root Cause:**

A duplicate declaration replaced the correct Open-Meteo endpoint with an empty string.

This would prevent the application from constructing a valid weather API request when using the default configuration.

**Correction:**

Remove the duplicate declaration and retain the correct endpoint:

```python
WEATHER_API_BASE_URL: str = "https://api.open-meteo.com/v1/forecast"
```

**Impact:**

The configuration now provides a valid weather API endpoint.

**Validation Status:** Effective configuration value verified at runtime.

---

### BUG-005 — Invalid Pydantic Configuration Attribute

**File:** components/config.py

**Category:** Configuration Initialization

**Original Code:**

```python
TEMP_MIN = str = "Needs Debugging"
```

**Root Cause:**

The declaration used chained assignment instead of a valid type annotation.

Pydantic requires model fields to contain appropriate type annotations.

**Observed Error:**

```text
pydantic.errors.PydanticUserError:
A non-annotated attribute was detected
```

**Correction:**

Remove the invalid and unnecessary `TEMP_MIN` declaration and eliminate its corresponding reference from the temperature classification function.

**Impact:**

Pydantic Settings can now initialize the application configuration correctly.

**Validation Status:** Verified through successful configuration initialization and subsequent application execution.

---

### BUG-006 — Incorrect Temperature Classification

**File:** components/helper_functions.py

**Category:** Business Logic

**Original Code:**

```python
if temp_celsius:
    return config.TEMP_MIN
```

**Root Cause:**

The conditional statement evaluated the truthiness of the temperature instead of comparing it against the configured temperature thresholds.

Almost every non-zero temperature would enter the first conditional branch, bypassing the intended classification logic.

**Correction:**

```python
def classify_temperature(temp_celsius: float) -> str:
    if temp_celsius < config.TEMP_COLD:
        return "cold"
    elif temp_celsius < config.TEMP_COOL:
        return "cool"
    elif temp_celsius < config.TEMP_COMFORTABLE:
        return "comfortable"
    elif temp_celsius < config.TEMP_WARM:
        return "warm"
    else:
        return "hot"
```

**Impact:**

Temperatures are now classified according to the configured thresholds.

**Validation Status:** 11 out of 11 functional test cases passed.

---

### BUG-007 — Location Data Not Stored in Agent State

**File:** components/nodes.py

**Category:** State Management

**Original Code:**

```python
state["location_data"] = {}
```

**Root Cause:**

The retrieved geographical information was discarded and replaced with an empty dictionary.

Consequently, subsequent nodes would not have access to the required geographical coordinates and location information.

**Correction:**

```python
state["location_data"] = location_data
```

**Impact:**

The corrected code preserves the retrieved location information in the shared agent state.

**Validation Status:** Verified through automated integration testing with mocked HTTP responses: location data is preserved and its coordinates are passed to the weather request. Live verification remains blocked by HTTP 429.

---

### BUG-008 — Incorrect Wind Speed Unit Formatting

**File:** components/nodes.py

**Category:** Output Formatting

**Original Code:**

```python
# wind_unit = units.get("windspeed", "km/h")
f"• Wind: {windspeed} wind_unit"
```

**Root Cause:**

The variable responsible for retrieving the wind speed unit was commented out.

Additionally, the final formatted string contained the literal text `wind_unit` instead of interpolating the corresponding variable.

**Correction:**

```python
wind_unit = units.get("windspeed", "km/h")
f"• Wind: {windspeed} {wind_unit}"
```

**Impact:**

The corrected code retrieves and displays the appropriate wind speed measurement unit, using km/h as the default when unit information is unavailable.

**Validation Status:** Verified through automated integration testing with both km/h and mph fixtures. The generated report uses the supplied unit dynamically.

### Bug Fix Summary

| Bug ID | Description | Correction Implemented | Validation |
|---|---|---|---|
| BUG-001 | Incorrect application entry point | Yes | Verified |
| BUG-002 | Disconnected weather retrieval node | Yes | Graph inspection and mocked integration verified |
| BUG-003 | Missing graph compilation | Yes | Verified |
| BUG-004 | Incorrect weather API endpoint | Yes | Configuration verified |
| BUG-005 | Invalid Pydantic configuration attribute | Yes | Verified |
| BUG-006 | Incorrect temperature classification | Yes | Functionally verified |
| BUG-007 | Location data discarded | Yes | Mocked integration verified |
| BUG-008 | Incorrect wind speed unit formatting | Yes | Mocked integration verified |

**Total primary bugs addressed:** 8

**Corrections individually verified:** 8

**Corrections awaiting runtime verification:** 2

---

### 5. Technical Improvements
In addition to the eight primary bug fixes, several technical improvements were implemented.

### IMP-001 — Improved Error Message

**File:** main.py

The original exception message incorrectly suggested that every application error was related to an internet connection problem.

The message was replaced with a more general instruction:

```python
print("Please review the error details and try again.")
```

This avoids attributing every exception to network connectivity.

### IMP-002 — Corrected Graph Execution Documentation

**File:** graph.py

Misleading comments about default parallel execution were replaced with comments describing the sequential workflow and graph dependencies.

### IMP-003 — Corrected Geographical Coordinate Documentation

**File:** components/nodes.py

The original comment incorrectly indicated that geographical coordinates required conversion from decimal degrees to radians.

Both APIs use geographical coordinates expressed in decimal degrees for the implemented requests.

The misleading comment was corrected.

### IMP-004 — Consistent Location Field Validation

**File:** components/nodes.py

The required location field was changed from:

```python
"country"
```

to:

```python
"country_name"
```

This aligns the field-presence validation with the location schema and the report generation logic.

### IMP-005 — Corrected API Authentication Documentation

**File:** components/nodes.py

The original comment incorrectly stated that an API key was required for the implemented Open-Meteo request.

The documentation was corrected to distinguish public non-commercial API access from other usage arrangements.

### IMP-006 — Explicit Dependency Declaration

**File:** requirements.txt

Two directly imported libraries were explicitly declared:

```text
requests==2.34.2
python-dotenv==1.2.4
```

Although these packages were already installed as transitive dependencies, declaring them explicitly improves dependency transparency and environment reproducibility.

### IMP-007 — Code Cleaning

Unused imports were removed from:

- config.py

- schema.py

- helper_functions.py

- nodes.py

The `timezone` import was subsequently restored because it became necessary for the improved timezone conversion logic.

### IMP-008 — Explicit GMT Reference Timezone

**File:** components/nodes.py

The weather API request was updated to explicitly specify GMT:

```python
params = {
    "latitude": location["latitude"],
    "longitude": location["longitude"],
    "current_weather": "true",
    "timezone": "GMT"
}
```

This establishes an explicit reference timezone for the subsequent local time conversion.

### IMP-009 — Improved Timezone Conversion

**File:** components/helper_functions.py

The original implementation calculated local time using direct datetime arithmetic.

The updated implementation normalizes datetime objects to UTC and uses `astimezone()` for timezone-aware conversion.

```python
utc_time = datetime.fromisoformat(
    utc_time_str.replace("Z", "+00:00")
)
if utc_time.tzinfo is None:
    utc_time = utc_time.replace(tzinfo=timezone.utc)
else:
    utc_time = utc_time.astimezone(timezone.utc)
offset = parse_utc_offset(utc_offset_str)
local_time = utc_time.astimezone(timezone(offset))
```

This improvement provides more consistent handling of timezone-aware and timezone-naive datetime objects.

### Technical Improvement Summary

| Improvement ID | Description |
|---|---|
| IMP-001 | Improved error messages |
| IMP-002 | Corrected graph execution documentation |
| IMP-003 | Corrected geographical coordinate documentation |
| IMP-004 | Consistent location field validation |
| IMP-005 | Corrected API authentication documentation |
| IMP-006 | Explicit dependency declaration |
| IMP-007 | Code cleaning |
| IMP-008 | Explicit GMT reference timezone |
| IMP-009 | Improved timezone conversion |

**Total technical improvements implemented:** 9

---

### 6. Functional Validation
The following five groups reproduce the 40 previously reviewed helper-function checks using explicit expected values. The time checks include an explicit UTC offset and midnight rollover. A failed assertion stops the cell.

In [15]:
temperature_cases = [(-5,"cold"),(0,"cold"),(9,"cold"),(10,"cool"),(17,"cool"),(18,"comfortable"),(25,"comfortable"),(26,"warm"),(31,"warm"),(32,"hot"),(40,"hot")]
weather_cases = [(0,"Clear sky"),(2,"Partly cloudy"),(3,"Overcast"),(45,"Fog"),(51,"Light drizzle"),(61,"Slight rain"),(75,"Heavy snow fall"),(95,"Thunderstorm"),(99,"Thunderstorm with heavy hail"),(999,"Weather code 999")]
greeting_cases = [(1,"Good morning"),(0,"Good evening"),(True,"Good morning"),(False,"Good evening")]
offset_cases = [("+05:30",5.5),("-06:00",-6),("+0530",5.5),("-0600",-6),("+00:00",0),("+02:45",2.75),("-03:30",-3.5),("+2",2),("invalid",0)]
time_cases = [(("2026-11-19T21:30","-0600"),"21:30 UTC | 15:30 (UTC-0600)"),(("2026-11-19T21:30Z","-0600"),"21:30 UTC | 15:30 (UTC-0600)"),(("2026-11-19T21:30+00:00","+0530"),"21:30 UTC | 03:00 (UTC+0530)"),(("2026-11-19T02:00","-0600"),"02:00 UTC | 20:00 (UTC-0600)"),(("2026-11-19T15:30-06:00","-0600"),"21:30 UTC | 15:30 (UTC-0600)"),(("invalid","-0600"),"Time unavailable")]
total = 0
for label, cases, fn in [
    ("Temperature", temperature_cases, classify_temperature),
    ("Weather description", weather_cases, get_weather_description),
    ("Greeting", greeting_cases, get_greeting),
    ("UTC offset", offset_cases, lambda x: parse_utc_offset(x).total_seconds()/3600),
    ("Time conversion", time_cases, lambda x: format_local_time(*x))
]:
    print(f"\n{label}")
    for value, expected in cases:
        actual = fn(value)
        assert actual == expected, (label, value, expected, actual)
        print(f"PASS | {value!r} -> {actual}")
        total += 1
print(f"\nFunctional checks passed: {total}/40")


Temperature
PASS | -5 -> cold
PASS | 0 -> cold
PASS | 9 -> cold
PASS | 10 -> cool
PASS | 17 -> cool
PASS | 18 -> comfortable
PASS | 25 -> comfortable
PASS | 26 -> warm
PASS | 31 -> warm
PASS | 32 -> hot
PASS | 40 -> hot

Weather description
PASS | 0 -> Clear sky
PASS | 2 -> Partly cloudy
PASS | 3 -> Overcast
PASS | 45 -> Fog
PASS | 51 -> Light drizzle
PASS | 61 -> Slight rain
PASS | 75 -> Heavy snow fall
PASS | 95 -> Thunderstorm
PASS | 99 -> Thunderstorm with heavy hail
PASS | 999 -> Weather code 999

Greeting
PASS | 1 -> Good morning
PASS | 0 -> Good evening
PASS | True -> Good morning
PASS | False -> Good evening

UTC offset
PASS | '+05:30' -> 5.5
PASS | '-06:00' -> -6.0
PASS | '+0530' -> 5.5
PASS | '-0600' -> -6.0
PASS | '+00:00' -> 0.0
PASS | '+02:45' -> 2.75
PASS | '-03:30' -> -3.5
PASS | '+2' -> 2.0
PASS | 'invalid' -> 0.0

Time conversion
PASS | ('2026-11-19T21:30', '-0600') -> 21:30 UTC | 15:30 (UTC-0600)
PASS | ('2026-11-19T21:30Z', '-0600') -> 21:30 UTC | 15:30 (UTC-0600)
P

### 7. Integration Testing
Run the existing unittest suite rather than copying its implementation. The suite intercepts HTTP calls but executes the real graph. It contains the successful workflow (16 checks), HTTP 429 handling (7 checks), and custom wind unit (2 checks).

In [16]:
suite = unittest.defaultTestLoader.discover(str(assignment_dir / "tests"), pattern="test_weather_agent.py")
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.testsRun == 3, f"Expected 3 tests, found {result.testsRun}"
assert result.wasSuccessful(), "Integration tests failed"
print("All three integration tests passed.")

test_location_api_rate_limit (test_weather_agent.TestWeatherAgent.test_location_api_rate_limit)
Test the workflow when IPAPI returns HTTP 429. ... ok
test_weather_agent_custom_wind_unit (test_weather_agent.TestWeatherAgent.test_weather_agent_custom_wind_unit)
Test dynamic wind unit formatting. ... ok
test_weather_agent_success (test_weather_agent.TestWeatherAgent.test_weather_agent_success)
Test the complete workflow using mocked API responses. ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.006s

OK
All three integration tests passed.


### 8. Weather Agent Demonstration
Display the actual generated report using the synthetic fixtures from the test module. The HTTP boundaries are mocked; the graph and nodes are unchanged.

In [17]:
sys.path.insert(0, str(assignment_dir / "tests"))
from test_weather_agent import MOCK_LOCATION_DATA, MOCK_WEATHER_DATA
from copy import deepcopy

def initial_state():
    return WeatherAgentState(name="Test User", location_data=None,
                             weather_data=None, weather_info=None)

def mock_response(payload):
    response = Mock(spec=requests.Response)
    response.status_code = 200
    response.raise_for_status.return_value = None
    response.json.return_value = deepcopy(payload)
    return response

with patch("components.nodes.requests.get", side_effect=[
    mock_response(MOCK_LOCATION_DATA), mock_response(MOCK_WEATHER_DATA)
]):
    final_state = weather_agent.invoke(initial_state())
print("SIMULATED WEATHER REPORT — NOT LIVE DATA")
print(final_state["weather_info"])

SIMULATED WEATHER REPORT — NOT LIVE DATA
Time: 18:00 UTC | 12:00 (UTC-0600)

Good morning, Test User!

Your current location: Mexico City, Mexico City, Mexico

Current weather conditions:
• Partly cloudy
• Temperature: 24.5°C (comfortable)
• Wind: 12.4 km/h


### 9. Error Handling
Reproduce the external HTTP 429 failure without using the network. The graph must raise the expected error and must not call the weather endpoint.

In [18]:
response = Mock(spec=requests.Response)
response.status_code = 429
response.raise_for_status.side_effect = requests.exceptions.HTTPError(
    "429 Client Error: Too Many Requests for url: https://ipapi.co/json/")
with patch("components.nodes.requests.get", return_value=response) as mocked_get:
    try:
        weather_agent.invoke(initial_state())
    except Exception as exc:
        assert "Failed to fetch location data" in str(exc) and "429" in str(exc)
        print("Expected simulated error:", exc)
    else:
        raise AssertionError("Expected HTTP 429 failure")
    mocked_get.assert_called_once_with(config.LOCATION_API_URL, timeout=config.REQUEST_TIMEOUT)
    response.json.assert_not_called()
print("PASS: error propagated; no weather request issued.")

Expected simulated error: Failed to fetch location data: 429 Client Error: Too Many Requests for url: https://ipapi.co/json/
PASS: error propagated; no weather request issued.


### 10. Conclusions
This assignment provided practical experience in analyzing, debugging, and validating a stateful application built with LangGraph and Python.

The debugging process demonstrated the importance of understanding application architecture, identifying root causes, preserving data dependencies, and performing systematic validation before declaring a solution complete.

### Key Achievements

- Reviewed the original application architecture and source code.

- Identified and addressed eight primary bugs.

- Implemented nine additional technical improvements.

- Corrected configuration and application initialization issues.

- Restored the required sequential graph execution path.

- Successfully compiled and inspected the LangGraph workflow.

- Improved temperature classification and timezone conversion.

- Explicitly declared directly imported dependencies.

- Verified all five auxiliary functions.

- Successfully completed 40 manually evaluated functional test cases.
- Passed three automated integration tests with 25 assertions and mock verification checks.
- Verified dynamic wind unit formatting and controlled HTTP 429 handling.
- Resolved three Pylance import warnings using local workspace configuration.

- Verified effective configuration values.

- Verified application initialization and graph invocation.

- Identified and documented an external API limitation.

### Final Assessment Status

| Metric | Result |
|---|---|
| Primary bugs addressed | 8/8 |
| Corrections individually verified (manual, structural, or mocked) | 8/8 |
| Technical improvements implemented | 9 |
| Auxiliary functions evaluated | 5/5 |
| Selected functional test cases passed | 40/40 |
| Selected manual check success rate | 100% |
| Automated integration tests | 3/3 passed |
| Automated checks | 25/25 |
| Combined successful checks | 65 |
| Automated test failures / errors | 0 / 0 |
| Pylance diagnostics | No problems detected |
| Dependency validation | Passed |
| Configuration validation | Passed |
| Graph validation | Passed |
| External incidents | 1 |
| Live end-to-end validation | Pending — HTTP 429 |

The implemented corrections and individual validation results demonstrate substantial progress in improving the application's reliability, correctness, and maintainability.

The corrected graph passes the selected mocked integration scenarios. Full live operational behavior cannot yet be certified because IPAPI rate limiting prevents completion of a real API workflow.

Live end-to-end validation must be repeated when the external service becomes available. The required `weather_agent_debug.ipynb` demonstration notebook remains a separate assessment deliverable; this README does not replace it.

This limitation has been explicitly documented to maintain technical transparency and avoid overstating the current validation results.

---

#### Known limitations
### INC-001 — External API Rate Limiting

| Attribute | Description |
|---|---|
| Incident ID | INC-001 |
| Provider | IPAPI |
| Component | fetch_location_data |
| HTTP Status | 429 |
| Error | Too Many Requests |
| Category | External Service Availability |
| Impact | Blocks end-to-end execution |
| Status | Unresolved |

The IPAPI service returned HTTP 429 during multiple application execution attempts.

This response indicates that the provider is limiting requests. The exact quota, rate-limiting policy applicable to the request, and recovery time have not been established.

The error is handled by the application. The remaining workflow was verified using mocked responses, but live behavior remains unverified until the location service becomes available.

The original API provider was preserved. Mocked responses are used exclusively in tests, not in the production workflow.

### LIM-001 — Retry Mechanism Not Implemented

The application configuration declares:

```python
MAX_RETRIES: int = 3
```

However, an effective retry mechanism is not currently implemented in the HTTP request logic.

Future improvements could include:

- Bounded retry attempts.

- Exponential backoff.

- Respect for Retry-After response headers.

- Handling appropriate transient HTTP failures.

- Improved logging.

The presence of the configuration parameter does not imply that retries are currently operational.

### LIM-002 — Invalid UTC Offset Fallback

The `parse_utc_offset()` function returns a zero-hour offset for certain malformed input strings.

While this prevents specific parsing errors from interrupting the application, it may cause invalid timezone information to be interpreted as UTC.

Stricter offset validation and explicit warning messages could improve data accuracy.

### LIM-003 — Greeting Granularity

The `get_greeting()` function uses a binary day/night indicator.

Consequently, it does not distinguish between morning and afternoon.

A future implementation could use the calculated local time to produce more precise greetings.

### LIM-004 — Fixed UTC Offset

The timezone conversion uses a fixed UTC offset obtained from the location service.

It does not independently calculate historical or future daylight saving time transitions using geographical timezone rules.

Python's `zoneinfo` module could be considered for a future implementation.

### LIM-005 — Calendar Date Not Displayed

The generated report displays UTC and local hours but does not display their corresponding calendar dates.

This can reduce clarity when a timezone conversion crosses midnight.

### LIM-006 — IP-Based Location Accuracy

The geographical information is derived from the outgoing IP address.

VPNs, proxies, shared networks, and internet service provider infrastructure may influence the reported location.

The result should be interpreted as an approximate geographical location rather than an exact physical position.

### LIM-007 — Data Validation Coverage

The current implementation checks the presence of required fields but does not comprehensively validate their types, ranges, or non-null values.

Although Pydantic data models are available in `schema.py`, they are not directly integrated into the current API response validation workflow.

### LIM-008 — Weather Code Coverage

The weather code dictionary contains a selection of WMO weather codes.

Unmapped codes are handled through a generic fallback description.

This mechanism was verified but does not guarantee complete meteorological code coverage.

### LIM-009 — External API Usage Restrictions

The application uses external services whose availability, quotas, and usage conditions are controlled by their respective providers.

Commercial deployment would require a review of the applicable API terms and licensing requirements.

### LIM-010 — Testing Coverage

The 40 successful functional test cases represent manually verified tests of selected auxiliary function behavior.

Three automated integration tests add 25 assertions and mock verification checks. These selected scenarios do not constitute exhaustive regression testing or measured code coverage.

The graph is verified under controlled HTTP responses, but live location and weather retrieval remain unverified. The automated tests invoke the graph directly and do not exercise the interactive `main.py` console interface end to end.

---

The notebook exercises the graph, not the interactive CLI entry point. Live service availability is not established by mocked tests. The historical 65 checks are not independent tests or a coverage metric. Keep generated cell outputs visible before submission. See the accompanying README for setup and detailed debugging history.